# Day 030 | Exploratory Data Analysis (EDA) Workflow

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** May 16, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| **Shape / dtypes / describe** | First look at data structure | 100-row site investigation table |
| **Missing value matrix** | Detect and quantify NaNs | Incomplete water content records |
| **Univariate distributions** | Histograms + normal overlay | N-value, Vs, water content |
| **Boxplots** | Visual outlier detection | Anomalous N-values, extreme unit weights |
| **Bivariate scatter** | Relationships between variables | N vs depth (coloured by soil type) |
| **Categorical analysis** | Count and mean per class | Mean N-value per soil type |
| **Correlation heatmap** | Pearson r matrix | N–Vs, depth–N relationships |
| **IQR + z-score outliers** | Systematic outlier flagging | Suspicious SPT readings |
| **Pair plot** | All pairwise scatters | Feature relationships for ML |
| **Data quality report** | Structured summary | Automated QC for site investigation |

---

## Our Geotechnical Problem

We have received a site investigation dataset from a ground improvement project: 100 samples from 6 boreholes (BH-01 to BH-06), each with SPT N-values, shear wave velocity (Vs), classification properties, and soil type labels. Before building any ML model for liquefaction prediction or bearing capacity estimation, we need a thorough EDA to:

1. Understand the data structure and distributions
2. Find and handle missing values and outliers
3. Identify relationships between geotechnical parameters
4. Produce a data quality report for the project record

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Optional: seaborn for enhanced visualisations
try:
    import seaborn as sns
    HAS_SEABORN = True
    sns.set_theme(style='whitegrid', palette='muted')
    print('seaborn available — using enhanced plots')
except ImportError:
    HAS_SEABORN = False
    print('seaborn not installed — using matplotlib only')

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

## Part 1: Load and Inspect the Dataset

In [ ]:
# ─────────────────────────────────────────────────────
# LOAD SITE INVESTIGATION DATA
# 100 samples from 6 boreholes
# ─────────────────────────────────────────────────────

df = pd.read_csv('site_investigation_eda.csv')

print('=== DATASET OVERVIEW ===')
print(f'Shape             : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Boreholes         : {sorted(df["borehole_id"].unique())}')
print(f'Soil types        : {sorted(df["soil_type"].unique())}')
print(f'Depth range       : {df["depth"].min():.1f} – {df["depth"].max():.1f} m')
print()

print('=== DATA TYPES ===')
print(df.dtypes)
print()

print('=== FIRST 5 ROWS ===')
df.head()

In [ ]:
# ─────────────────────────────────────────────────────
# df.info() and describe() — the EDA starting point
# ─────────────────────────────────────────────────────

print('=== df.info() ===')
df.info()
print()

# Ensure numeric columns are properly typed (some may have been read as object due to missing values)
numeric_cols = ['depth', 'N_value', 'Vs', 'water_content', 'liquid_limit',
                'plasticity_index', 'unit_weight', 'void_ratio', 'fines_content']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('=== DESCRIPTIVE STATISTICS ===')
df[numeric_cols].describe().round(2)

## Part 2: Missing Value Analysis

In [ ]:
# ─────────────────────────────────────────────────────
# MISSING VALUE MATRIX
# Count, percentage, and visual pattern of NaNs
# ─────────────────────────────────────────────────────

missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count'  : missing_count,
    'Missing %'      : missing_pct,
    'Data Type'      : df.dtypes,
    'Non-Null Count' : df.notnull().sum()
})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('=== MISSING VALUE REPORT ===')
if len(missing_df) == 0:
    print('  No missing values found.')
else:
    print(missing_df.to_string())

# Visual: missing value matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of missing %
all_pct = (df.isnull().sum() / len(df) * 100)
colors   = ['crimson' if v > 0 else 'steelblue' for v in all_pct]
axes[0].bar(all_pct.index, all_pct.values, color=colors)
axes[0].axhline(5, color='orange', ls='--', lw=1.5, label='5% threshold')
axes[0].set_xticklabels(all_pct.index, rotation=45, ha='right')
axes[0].set_ylabel('Missing (%)')
axes[0].set_title('Missing Values per Column')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Heatmap of null pattern
null_matrix = df[numeric_cols].isnull().astype(int)
axes[1].imshow(null_matrix.T, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')
axes[1].set_yticks(range(len(numeric_cols)))
axes[1].set_yticklabels(numeric_cols)
axes[1].set_xlabel('Sample index')
axes[1].set_title('Null Pattern (red = missing)')

plt.tight_layout()
plt.show()

print(f'\nTotal cells     : {df.shape[0] * df.shape[1]}')
print(f'Total missing   : {df.isnull().sum().sum()}')
print(f'Overall missing : {df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%')

## Part 3: Univariate Distributions — Histograms with Normal Overlay

In [ ]:
# ─────────────────────────────────────────────────────
# HISTOGRAMS: N_value, Vs, water_content
# Normal curve overlay to check distributional assumption
# (ML models like Gaussian NB assume normality)
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

params = [
    ('N_value',      'SPT N-value',           'steelblue',    'count'),
    ('Vs',           'Vs (m/s)',               'darkorange',   'count'),
    ('water_content','Water Content (%)',      'green',        'count'),
]

for ax, (col, label, color, _) in zip(axes, params):
    data = df[col].dropna()
    mu, sigma = data.mean(), data.std()

    ax.hist(data, bins=15, color=color, alpha=0.65, edgecolor='white',
            density=True, label='Data histogram')

    # Normal curve overlay
    x_norm = np.linspace(data.min(), data.max(), 200)
    ax.plot(x_norm, stats.norm.pdf(x_norm, mu, sigma),
            color='black', lw=2, label=f'Normal(μ={mu:.1f}, σ={sigma:.1f})')

    ax.axvline(mu,        color='red',  ls='--', lw=1.5, label=f'Mean={mu:.1f}')
    ax.axvline(mu-sigma,  color='gray', ls=':',  lw=1.2)
    ax.axvline(mu+sigma,  color='gray', ls=':',  lw=1.2, label='±1σ')

    sk = stats.skew(data)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'{label}\nskewness = {sk:.2f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Univariate Distributions — Key Geotechnical Parameters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Normality check (Shapiro-Wilk test):')
for col, label, *_ in params:
    data = df[col].dropna()
    stat, p = stats.shapiro(data)
    normal = 'NORMAL (p>0.05)' if p > 0.05 else 'NOT normal (p≤0.05)'
    print(f'  {label:<25}: W={stat:.4f}, p={p:.4f}  → {normal}')

## Part 4: Boxplots — Outlier Detection

In [ ]:
# ─────────────────────────────────────────────────────
# BOXPLOTS: N_value, unit_weight, void_ratio
# Box whiskers = 1.5×IQR; points beyond = potential outliers
# ─────────────────────────────────────────────────────

box_cols   = ['N_value', 'unit_weight', 'void_ratio', 'Vs']
box_labels = ['SPT N-value', 'Unit Weight (kN/m³)', 'Void Ratio', 'Vs (m/s)']
box_colors = ['steelblue', 'darkorange', 'green', 'purple']

fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for ax, col, label, color in zip(axes, box_cols, box_labels, box_colors):
    data = df[col].dropna()

    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.5),
                    medianprops=dict(color='black', lw=2),
                    flierprops=dict(marker='o', markerfacecolor='red',
                                    markersize=8, alpha=0.7))

    # IQR statistics
    Q1, Q3  = data.quantile(0.25), data.quantile(0.75)
    IQR     = Q3 - Q1
    lower   = Q1 - 1.5 * IQR
    upper   = Q3 + 1.5 * IQR
    n_out   = ((data < lower) | (data > upper)).sum()

    ax.set_title(f'{label}\nOutliers: {n_out}')
    ax.set_ylabel(label)
    ax.text(1.3, data.median(), f'Median={data.median():.1f}',
            va='center', fontsize=8, color='navy')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Boxplots — Outlier Detection in Geotechnical Parameters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Outlier summary (IQR method, 1.5×IQR fence):')
for col, label in zip(box_cols, box_labels):
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    out = data[(data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)]
    print(f'  {label:<25}: {len(out)} outlier(s)  '
          f'[Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}, '
          f'fence: {Q1-1.5*IQR:.1f}–{Q3+1.5*IQR:.1f}]')

## Part 5: Bivariate Analysis — N vs Depth and N vs Vs

In [ ]:
# ─────────────────────────────────────────────────────
# BIVARIATE SCATTER PLOTS
# (1) N vs depth — colour-coded by soil_type
# (2) N vs Vs — with regression line
# ─────────────────────────────────────────────────────

soil_colors = {
    'Sand'  : 'goldenrod',
    'Silt'  : 'steelblue',
    'Clay'  : 'saddlebrown',
    'Gravel': 'gray',
    'Fill'  : 'crimson'
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Plot 1: N vs Depth (colour by soil type) ---
for soil, color in soil_colors.items():
    mask = df['soil_type'] == soil
    axes[0].scatter(df.loc[mask, 'N_value'], df.loc[mask, 'depth'],
                    c=color, s=45, alpha=0.75, edgecolors='white', lw=0.5,
                    label=soil)

# Trend line through all data
valid = df[['N_value', 'depth']].dropna()
z = np.polyfit(valid['N_value'], valid['depth'], 1)
p = np.poly1d(z)
N_range = np.linspace(valid['N_value'].min(), valid['N_value'].max(), 100)
axes[0].plot(N_range, p(N_range), 'k--', lw=1.5, alpha=0.6, label='Linear trend')

axes[0].invert_yaxis()
axes[0].set_xlabel('SPT N-value')
axes[0].set_ylabel('Depth (m)')
axes[0].set_title('N-value vs Depth (coloured by soil type)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Plot 2: N vs Vs (with regression) ---
valid2 = df[['N_value', 'Vs', 'soil_type']].dropna()
for soil, color in soil_colors.items():
    mask = valid2['soil_type'] == soil
    axes[1].scatter(valid2.loc[mask, 'N_value'], valid2.loc[mask, 'Vs'],
                    c=color, s=45, alpha=0.75, edgecolors='white', lw=0.5,
                    label=soil)

# Power-law regression: Vs = a * N^b (Imai & Tonouchi style)
log_N  = np.log(valid2['N_value'].values)
log_Vs = np.log(valid2['Vs'].values)
b_fit, log_a_fit = np.polyfit(log_N, log_Vs, 1)
a_fit = np.exp(log_a_fit)

N_fit_range = np.linspace(valid2['N_value'].min(), valid2['N_value'].max(), 100)
Vs_fit = a_fit * N_fit_range**b_fit
axes[1].plot(N_fit_range, Vs_fit, 'k-', lw=2,
             label=f'Power-law: Vs={a_fit:.0f}·N^{b_fit:.3f}')

# Compute R²
Vs_pred = a_fit * valid2['N_value'].values**b_fit
ss_res  = np.sum((valid2['Vs'].values - Vs_pred)**2)
ss_tot  = np.sum((valid2['Vs'].values - valid2['Vs'].mean())**2)
r2      = 1 - ss_res / ss_tot

axes[1].set_xlabel('SPT N-value')
axes[1].set_ylabel('Vs (m/s)')
axes[1].set_title(f'N-value vs Vs — Power-Law Fit (R²={r2:.3f})')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bivariate Analysis — Geotechnical Parameter Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

r_NVs = df[['N_value', 'Vs']].dropna().corr().iloc[0, 1]
r_Ndepth = df[['N_value', 'depth']].dropna().corr().iloc[0, 1]
print(f'Pearson r (N vs Vs)   : {r_NVs:.3f}')
print(f'Pearson r (N vs depth): {r_Ndepth:.3f}')
print(f'Power-law fit         : Vs = {a_fit:.1f} × N^{b_fit:.3f}  (R²={r2:.3f})')

## Part 6: Categorical Analysis — Soil Type Counts and Mean N per Type

In [ ]:
# ─────────────────────────────────────────────────────
# CATEGORICAL ANALYSIS
# Bar chart of soil_type counts
# Mean N-value per soil type (geotechnical interpretation)
# ─────────────────────────────────────────────────────

soil_counts  = df['soil_type'].value_counts()
mean_N_soil  = df.groupby('soil_type')['N_value'].agg(['mean', 'std', 'count']).round(1)
mean_N_soil  = mean_N_soil.sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: soil type counts
colors_list = [soil_colors.get(s, 'gray') for s in soil_counts.index]
axes[0].bar(soil_counts.index, soil_counts.values, color=colors_list,
            edgecolor='white', lw=1.5)
for i, (s, v) in enumerate(zip(soil_counts.index, soil_counts.values)):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Soil Type')
axes[0].set_ylabel('Sample Count')
axes[0].set_title('Sample Count per Soil Type')
axes[0].grid(True, alpha=0.3, axis='y')

# Bar chart: mean N per soil type with error bars
colors_N = [soil_colors.get(s, 'gray') for s in mean_N_soil.index]
bars = axes[1].bar(mean_N_soil.index, mean_N_soil['mean'],
                   yerr=mean_N_soil['std'],
                   color=colors_N, edgecolor='white', lw=1.5,
                   capsize=5, error_kw={'elinewidth': 1.5, 'ecolor': 'black'})
for i, (idx, row) in enumerate(mean_N_soil.iterrows()):
    axes[1].text(i, row['mean'] + row['std'] + 1,
                 f"{row['mean']:.0f}\n(n={row['count']:.0f})",
                 ha='center', fontsize=9)
axes[1].set_xlabel('Soil Type')
axes[1].set_ylabel('Mean N-value')
axes[1].set_title('Mean SPT N-value per Soil Type (±1σ)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Categorical Analysis — Soil Type Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Mean N-value by soil type:')
print(mean_N_soil.to_string())
print(f'\nSaturation rate per soil type:')
print(df.groupby('soil_type')['is_saturated'].mean().round(2).to_string())

## Part 7: Correlation Heatmap

In [ ]:
# ─────────────────────────────────────────────────────
# CORRELATION HEATMAP — numerical columns only
# Pearson r with significance annotation
# ─────────────────────────────────────────────────────

corr_cols = ['depth', 'N_value', 'Vs', 'water_content', 'liquid_limit',
             'plasticity_index', 'unit_weight', 'void_ratio', 'fines_content']
corr_df   = df[corr_cols].dropna()
corr_mat  = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))

# Manual heatmap with annotations
n = len(corr_cols)
im = ax.imshow(corr_mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.8)

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)

for i in range(n):
    for j in range(n):
        val = corr_mat.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=color, fontweight='bold' if abs(val) > 0.6 else 'normal')

ax.set_title('Correlation Matrix — Site Investigation Parameters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print strongest correlations
corr_pairs = []
for i in range(n):
    for j in range(i+1, n):
        r  = corr_mat.values[i, j]
        corr_pairs.append((corr_cols[i], corr_cols[j], r))
corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print('Top 8 strongest correlations:')
for c1, c2, r in corr_pairs[:8]:
    direction = 'positive' if r > 0 else 'negative'
    strength  = 'strong' if abs(r) > 0.6 else ('moderate' if abs(r) > 0.4 else 'weak')
    print(f'  {c1:<20} ↔ {c2:<20}: r={r:+.3f}  ({strength} {direction})')

## Part 8: Outlier Detection — IQR Method and Z-Score

In [ ]:
# ─────────────────────────────────────────────────────
# OUTLIER DETECTION ON N_value
# Method 1: IQR fence (Tukey, 1977)
# Method 2: Z-score threshold (|z| > 3)
# ─────────────────────────────────────────────────────

col = 'N_value'
data_n = df[[col, 'borehole_id', 'depth', 'soil_type']].dropna().copy()

# --- IQR method ---
Q1  = data_n[col].quantile(0.25)
Q3  = data_n[col].quantile(0.75)
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR
data_n['iqr_outlier'] = (data_n[col] < lower_iqr) | (data_n[col] > upper_iqr)

# --- Z-score method ---
data_n['z_score'] = np.abs(stats.zscore(data_n[col]))
data_n['z_outlier'] = data_n['z_score'] > 3.0

# --- Visualise ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# IQR plot
normal_iqr  = data_n[~data_n['iqr_outlier']]
outlier_iqr = data_n[data_n['iqr_outlier']]
axes[0].scatter(normal_iqr.index,  normal_iqr[col],  s=40, color='steelblue', alpha=0.7, label='Normal')
axes[0].scatter(outlier_iqr.index, outlier_iqr[col], s=90, color='red',       alpha=0.9,
                marker='*', label=f'IQR outlier (n={len(outlier_iqr)})')
axes[0].axhline(upper_iqr, color='orange', ls='--', lw=1.5, label=f'Upper fence={upper_iqr:.1f}')
axes[0].axhline(lower_iqr, color='orange', ls='--', lw=1.5, label=f'Lower fence={lower_iqr:.1f}')
axes[0].set_xlabel('Sample Index'); axes[0].set_ylabel('N-value')
axes[0].set_title(f'IQR Outlier Detection — N_value')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Z-score plot
normal_z  = data_n[~data_n['z_outlier']]
outlier_z = data_n[data_n['z_outlier']]
axes[1].scatter(normal_z.index,  normal_z[col],  s=40, color='steelblue', alpha=0.7, label='Normal')
axes[1].scatter(outlier_z.index, outlier_z[col], s=90, color='red',       alpha=0.9,
                marker='*', label=f'Z-score outlier (n={len(outlier_z)})')
axes[1].axhline(data_n[col].mean() + 3*data_n[col].std(),
                color='purple', ls='--', lw=1.5, label='μ + 3σ')
axes[1].axhline(data_n[col].mean() - 3*data_n[col].std(),
                color='purple', ls='--', lw=1.5, label='μ − 3σ')
axes[1].set_xlabel('Sample Index'); axes[1].set_ylabel('N-value')
axes[1].set_title('Z-Score Outlier Detection — N_value')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Outlier Detection: IQR vs Z-Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('IQR outliers detected:')
if len(outlier_iqr) > 0:
    print(outlier_iqr[['borehole_id', 'depth', col, 'soil_type', 'iqr_outlier']].to_string(index=False))
print(f'\nZ-score outliers (|z| > 3): {len(outlier_z)} found')
if len(outlier_z) > 0:
    print(outlier_z[['borehole_id', 'depth', col, 'z_score', 'soil_type']].to_string(index=False))

## Part 9: Pair Plot

In [ ]:
# ─────────────────────────────────────────────────────
# PAIR PLOT: N_value, Vs, water_content, unit_weight
# Use seaborn if available; fallback to matplotlib
# ─────────────────────────────────────────────────────

pair_cols = ['N_value', 'Vs', 'water_content', 'unit_weight']
pair_df   = df[pair_cols + ['soil_type']].dropna()

if HAS_SEABORN:
    palette = {k: v for k, v in soil_colors.items() if k in pair_df['soil_type'].unique()}
    g = sns.pairplot(pair_df, hue='soil_type', palette=palette,
                     diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30})
    g.figure.suptitle('Pair Plot — N_value, Vs, water_content, unit_weight',
                       y=1.02, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    # Manual pair plot fallback
    n_p = len(pair_cols)
    fig, axes = plt.subplots(n_p, n_p, figsize=(13, 12))
    for i, col_i in enumerate(pair_cols):
        for j, col_j in enumerate(pair_cols):
            ax = axes[i][j]
            if i == j:
                ax.hist(pair_df[col_i].dropna(), bins=12,
                        color='steelblue', alpha=0.7, edgecolor='white')
                ax.set_xlabel(col_i)
            else:
                for soil, color in soil_colors.items():
                    m = pair_df['soil_type'] == soil
                    ax.scatter(pair_df.loc[m, col_j], pair_df.loc[m, col_i],
                               s=15, alpha=0.6, color=color)
                ax.set_xlabel(col_j)
                ax.set_ylabel(col_i)
            ax.tick_params(labelsize=7)
    plt.suptitle('Pair Plot — Geotechnical Parameters', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('Pair plot key relationships:')
for i, c1 in enumerate(pair_cols):
    for c2 in pair_cols[i+1:]:
        r = pair_df[[c1, c2]].corr().iloc[0, 1]
        print(f'  {c1:<15} ↔ {c2:<15}: r = {r:+.3f}')

## Part 10: EDA Summary — 5 Key Findings

In [ ]:
# ─────────────────────────────────────────────────────
# STRUCTURED EDA SUMMARY: 5 KEY FINDINGS
# ─────────────────────────────────────────────────────

# Compute summary statistics for the report
n_total     = len(df)
n_missing   = df.isnull().sum().sum()
miss_pct    = n_missing / (n_total * df.shape[1]) * 100

Q1_N, Q3_N  = df['N_value'].quantile([0.25, 0.75])
IQR_N       = Q3_N - Q1_N
n_iqr_out   = ((df['N_value'] < Q1_N - 1.5*IQR_N) | (df['N_value'] > Q3_N + 1.5*IQR_N)).sum()

r_N_Vs      = df[['N_value', 'Vs']].corr().iloc[0, 1]
r_N_wc      = df[['N_value', 'water_content']].dropna().corr().iloc[0, 1]

dominant_soil = df['soil_type'].value_counts().index[0]
weakest_soil  = df.groupby('soil_type')['N_value'].mean().idxmin()
strongest_soil = df.groupby('soil_type')['N_value'].mean().idxmax()

print('=' * 65)
print(' EDA SUMMARY — SITE INVESTIGATION DATASET')
print(' 5 KEY GEOTECHNICAL FINDINGS')
print('=' * 65)

findings = [
    (
        "1. Data Completeness",
        f"   {n_total} samples from 6 boreholes. {n_missing} missing values "
        f"({miss_pct:.1f}% overall). Missing values in water_content, "
        f"liquid_limit, and Vs columns. Imputation required before ML."
    ),
    (
        "2. N-value Outliers",
        f"   {n_iqr_out} outliers detected by IQR method in N_value. "
        f"These may represent anomalous layers (cemented, bouldery) or "
        f"erroneous measurements. Flag for engineer review before ML training."
    ),
    (
        "3. Strong N–Vs Correlation",
        f"   Pearson r(N, Vs) = {r_N_Vs:.3f} — strong positive correlation. "
        f"The power-law Vs = a·N^b (Imai & Tonouchi type) fits well (R²={r2:.3f}). "
        f"Vs can be predicted from SPT N without additional testing."
    ),
    (
        "4. Soil Type Controls Strength",
        f"   {strongest_soil} has highest mean N ({df.groupby('soil_type')['N_value'].mean()[strongest_soil]:.1f}), "
        f"while {weakest_soil} has the lowest ({df.groupby('soil_type')['N_value'].mean()[weakest_soil]:.1f}). "
        f"Soil type is a critical feature for any ML classification or regression model."
    ),
    (
        "5. N–Water Content Trend",
        f"   r(N, water_content) = {r_N_wc:.3f} — negative correlation: "
        f"higher water content soils (clay, silt) tend to have lower N-values. "
        f"Both parameters are informative for liquefaction and bearing capacity models."
    ),
]

for title, detail in findings:
    print(f'\n{title}:')
    print(detail)

print('\n' + '=' * 65)

## Part 11: Automated Data Quality Report

In [ ]:
# ═══════════════════════════════════════════════════════
# AUTOMATED DATA QUALITY REPORT
# Structured printout for the project record
# ═══════════════════════════════════════════════════════

def geotechnical_data_quality_report(df, numeric_cols, label='Site Investigation'):
    """Print a structured data quality report for geotechnical datasets."""
    print('=' * 68)
    print(f'  DATA QUALITY REPORT — {label.upper()}')
    print('  Generated for ML preprocessing')
    print('=' * 68)

    # 1. Shape
    print(f'\n[ DATASET SHAPE ]')
    print(f'  Rows: {df.shape[0]}   Columns: {df.shape[1]}')
    print(f'  Numerical: {len(numeric_cols)}   Categorical: {df.shape[1] - len(numeric_cols)}')

    # 2. Missing values
    print(f'\n[ MISSING VALUES ]')
    for col in df.columns:
        n_miss = df[col].isnull().sum()
        if n_miss > 0:
            pct = n_miss / len(df) * 100
            rec = 'Drop' if pct > 30 else ('Impute median' if col in numeric_cols else 'Impute mode')
            print(f'  {col:<22}: {n_miss:3d} missing ({pct:5.1f}%)  → Recommendation: {rec}')
        else:
            print(f'  {col:<22}: OK (no missing)')

    # 3. Outliers
    print(f'\n[ OUTLIERS (IQR 1.5× fence) ]')
    for col in numeric_cols:
        data = df[col].dropna()
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        n_out = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
        pct_out = n_out / len(data) * 100
        status = 'REVIEW' if n_out > 0 else 'OK'
        print(f'  {col:<22}: {n_out:2d} outliers ({pct_out:4.1f}%)  [{status}]')

    # 4. Distributional checks
    print(f'\n[ DISTRIBUTIONAL CHECKS (Shapiro-Wilk, n≤50) ]')
    sample = df[numeric_cols].dropna()
    n_sw = min(len(sample), 50)
    sample_sw = sample.sample(n=n_sw, random_state=42) if len(sample) > 50 else sample
    for col in numeric_cols:
        sk  = stats.skew(sample[col])
        ku  = stats.kurtosis(sample[col])
        _, p = stats.shapiro(sample_sw[col])
        normal = 'Normal' if p > 0.05 else 'Non-normal'
        xform  = '  → Consider log-transform' if abs(sk) > 1.5 else ''
        print(f'  {col:<22}: skew={sk:+.2f}, kurt={ku:+.2f}  {normal}{xform}')

    # 5. Categorical columns
    cat_cols = [c for c in df.columns if c not in numeric_cols and df[c].dtype == object]
    print(f'\n[ CATEGORICAL COLUMNS ]')
    for col in cat_cols:
        vc = df[col].value_counts()
        print(f'  {col:<22}: {df[col].nunique()} unique values — {dict(vc.head(5))}')

    # 6. ML readiness
    print(f'\n[ ML READINESS CHECKLIST ]')
    total_miss = df.isnull().sum().sum()
    checks = [
        ('Missing values resolved', total_miss == 0),
        ('Target variable present', 'N_value' in df.columns),
        ('Categorical encoding needed', len(cat_cols) > 0),
        ('Feature scaling recommended', True),
        ('Outlier treatment needed', any(
            ((df[c].dropna() < df[c].dropna().quantile(0.25) - 1.5*(df[c].dropna().quantile(0.75)-df[c].dropna().quantile(0.25))) |
             (df[c].dropna() > df[c].dropna().quantile(0.75) + 1.5*(df[c].dropna().quantile(0.75)-df[c].dropna().quantile(0.25)))).any()
            for c in numeric_cols)),
    ]
    for desc, status in checks:
        icon = '[YES]' if status else '[NO] '
        print(f'  {icon} {desc}')

    print('\n' + '=' * 68)
    print('  Report complete. Data is ready for preprocessing pipeline.')
    print('=' * 68)


geotechnical_data_quality_report(df, numeric_cols, label='BH-01 to BH-06 Site Investigation')

## Practice Exercises

### Exercise 1
Impute missing values: use median imputation for `water_content` and `liquid_limit`, and KNN imputation for `Vs`. Compare the distributions before and after imputation.

### Exercise 2
Create a new feature `liquefaction_susceptible` (boolean) based on: N_value < 20, fines_content < 35, depth < 10 m, and is_saturated == True. Plot its distribution per soil type.

### Exercise 3
Use `pd.cut` to bin `depth` into three layers (0–6 m, 6–12 m, 12+ m). Compute mean N-value per layer and per soil type, producing a pivot table.

```python
# Exercise 1 starter
# from sklearn.impute import KNNImputer
# df['water_content'].fillna(df['water_content'].median(), inplace=True)
# imputer = KNNImputer(n_neighbors=5)
# df[['Vs']] = imputer.fit_transform(df[['N_value', 'depth', 'Vs']])[:, [-1]]

# Exercise 2 starter
# df['liq_susceptible'] = (
#     (df['N_value'] < 20) & (df['fines_content'] < 35) &
#     (df['depth'] < 10) & (df['is_saturated'] == True)
# )

# Exercise 3 starter
# df['depth_layer'] = pd.cut(df['depth'], bins=[0, 6, 12, 20],
#                             labels=['Shallow', 'Intermediate', 'Deep'])
# pivot = df.pivot_table(values='N_value', index='depth_layer',
#                        columns='soil_type', aggfunc='mean').round(1)
```

---

## Key Takeaways

1. **Start with shape, dtypes, and describe** before any visualisation — know your data types, ranges, and obvious anomalies first.
2. **Missing values are engineering signals**: missing Vs often means no MASW was done, not that Vs is zero. Document the reason before imputing.
3. **IQR + z-score together**: IQR catches moderate outliers (1.5–3σ range); z-score > 3 catches extreme ones. Flag all for engineer review — don't silently drop.
4. **Correlation is not causation**: N and depth correlate positively because deeper soils are generally denser, not because depth *causes* high N. Soil type is a confounding variable.
5. **EDA shapes your ML model**: skewed features need log-transforms, categorical features need encoding, outliers may need Winsorisation — EDA tells you exactly what preprocessing is required.

## How This Connects to ML

| Today | In ML (Phase 3–5) |
|-------|-------------------|
| Histogram / normality check | Decide on log-transform before Linear Regression |
| Correlation heatmap | Feature selection / VIF analysis for regression |
| Outlier detection | Training data cleaning before tree models |
| Missing value report | Imputation strategy (KNN, median, MICE) |
| Pair plot | Identify redundant features for PCA / dimensionality reduction |
| Categorical counts | Check class balance before classification training |

---

*Day 030 of 100 — A third of the way there! You now know your data as well as a geotechnical engineer knows the soil profile!*